# Exercícios Práticos - Processamento de Linguagem Natural (PLN)
**Fatec Jahu**

Desenvolva as soluções para cada um dos exercícios utilizando as células de código abaixo de cada enunciado. Para as questões teóricas, você pode escrever sua resposta como comentários no código ou converter a célula para Markdown.

### Exercício 1: Parametrização da Gravação
Modifique a função `gravar_audio` para que o usuário possa definir a taxa de amostragem (`fs`), o número de canais e o tempo de gravação dinamicamente no momento da execução, em vez de depender apenas dos valores padrão. Adicione um tratamento de erro (`try/except`) para caso o microfone não seja detectado.

In [ ]:
import sounddevice as sd
import scipy.io.wavfile as wav
import time

def gravar_audio(nome_arquivo="audio.wav", duracao=None, fs=None, canais=None):
    """
    Grava áudio do microfone com parâmetros definidos dinamicamente pelo usuário.
    Se os parâmetros não forem fornecidos, solicita ao usuário via input.
    """
    # Solicitar parâmetros ao usuário caso não sejam fornecidos
    if fs is None:
        try:
            fs = int(input("Digite a taxa de amostragem (ex: 44100): ") or 44100)
        except ValueError:
            print("Valor inválido. Usando taxa padrão de 44100 Hz.")
            fs = 44100

    if canais is None:
        try:
            canais = int(input("Digite o número de canais (1=mono, 2=estéreo): ") or 1)
        except ValueError:
            print("Valor inválido. Usando 1 canal (mono).")
            canais = 1

    if duracao is None:
        try:
            duracao = int(input("Digite o tempo de gravação em segundos (ex: 5): ") or 5)
        except ValueError:
            print("Valor inválido. Usando duração padrão de 5 segundos.")
            duracao = 5

    print(f"\nConfiguração: fs={fs} Hz | canais={canais} | duração={duracao}s")
    print("Preparando para gravar...")

    # Contador regressivo
    for i in range(3, 0, -1):
        print(i)
        time.sleep(1)

    try:
        print("Gravando...")
        audio = sd.rec(int(duracao * fs), samplerate=fs, channels=canais)
        sd.wait()

        wav.write(nome_arquivo, fs, audio)
        print("Gravação finalizada!")

    except sd.PortAudioError:
        print("Erro: Microfone não detectado. Verifique se há um dispositivo de áudio conectado.")
    except Exception as e:
        print(f"Erro inesperado durante a gravação: {e}")

# Teste com parâmetros fixos (para não bloquear com input)
print("Função gravar_audio definida com sucesso!")
print("Para testar, execute: gravar_audio(duracao=3, fs=44100, canais=1)")

### Exercício 2: Visualização de Dados Acústicos
Antes de enviar o áudio para a transcrição, carregue o arquivo `audio.wav` utilizando bibliotecas como `numpy` e `matplotlib` e plote o gráfico de forma de onda (waveform) do áudio gravado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.io.wavfile as wav
import os

def visualizar_waveform(arquivo="audio.wav"):
    """
    Carrega um arquivo WAV e plota o gráfico de forma de onda (waveform).
    """
    if not os.path.exists(arquivo):
        print(f"Arquivo '{arquivo}' não encontrado. Gerando áudio de exemplo para demonstração...")
        # Gerar um áudio de exemplo (tom senoidal de 440 Hz por 2 segundos)
        fs = 44100
        duracao = 2
        t = np.linspace(0, duracao, int(fs * duracao), endpoint=False)
        # Combinação de frequências para waveform mais interessante
        audio_exemplo = 0.5 * np.sin(2 * np.pi * 440 * t) + 0.3 * np.sin(2 * np.pi * 880 * t)
        audio_exemplo = (audio_exemplo * 32767).astype(np.int16)
        wav.write(arquivo, fs, audio_exemplo)
        print(f"Áudio de exemplo gerado: {arquivo}")

    # Carregar o arquivo de áudio
    fs, dados = wav.read(arquivo)

    # Se estéreo, pegar apenas o primeiro canal para visualização
    if len(dados.shape) > 1:
        dados = dados[:, 0]

    # Criar o eixo de tempo
    duracao_total = len(dados) / fs
    tempo = np.linspace(0, duracao_total, num=len(dados))

    # Plotar o gráfico
    plt.figure(figsize=(14, 5))
    plt.plot(tempo, dados, color='#2196F3', linewidth=0.5, alpha=0.8)
    plt.fill_between(tempo, dados, alpha=0.15, color='#2196F3')
    plt.title('Forma de Onda (Waveform) do Áudio', fontsize=14, fontweight='bold')
    plt.xlabel('Tempo (s)', fontsize=12)
    plt.ylabel('Amplitude', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print(f"\nInformações do áudio:")
    print(f"  Taxa de amostragem: {fs} Hz")
    print(f"  Duração: {duracao_total:.2f} segundos")
    print(f"  Amostras: {len(dados)}")

visualizar_waveform("audio.wav")

### Exercício 3: Análise de Modelos Whisper
O código original utiliza o modelo `"base"` do Whisper. Altere o código para carregar os modelos `"tiny"` e `"small"`. Execute testes de fala e documente (em comentários) as diferenças observadas em relação à precisão do reconhecimento do texto e ao tempo de processamento necessário para cada um.

In [ ]:
import whisper
import time
import os

def testar_modelos_whisper(arquivo="audio.wav"):
    """
    Carrega e testa os modelos 'tiny', 'base' e 'small' do Whisper,
    documentando diferenças de precisão e tempo de processamento.
    """
    if not os.path.exists(arquivo):
        print(f"Arquivo '{arquivo}' não encontrado.")
        print("Execute a gravação de áudio antes de testar os modelos.")
        return

    modelos = ["tiny", "base", "small"]
    resultados = {}

    for nome_modelo in modelos:
        print(f"\n{'='*50}")
        print(f"Testando modelo: {nome_modelo.upper()}")
        print(f"{'='*50}")

        # Medir tempo de carregamento do modelo
        inicio_carga = time.time()
        modelo = whisper.load_model(nome_modelo)
        tempo_carga = time.time() - inicio_carga
        print(f"Tempo de carregamento: {tempo_carga:.2f}s")

        # Medir tempo de transcrição
        inicio_transcricao = time.time()
        resultado = modelo.transcribe(arquivo, language="pt")
        tempo_transcricao = time.time() - inicio_transcricao

        texto = resultado["text"]
        print(f"Tempo de transcrição: {tempo_transcricao:.2f}s")
        print(f"Texto reconhecido: {texto}")

        resultados[nome_modelo] = {
            "texto": texto,
            "tempo_carga": tempo_carga,
            "tempo_transcricao": tempo_transcricao
        }

    # Resumo comparativo
    print(f"\n{'='*60}")
    print("RESUMO COMPARATIVO DOS MODELOS")
    print(f"{'='*60}")
    print(f"{'Modelo':<10} {'Carga (s)':<12} {'Transcrição (s)':<18} {'Texto'}")
    print(f"{'-'*60}")
    for nome, dados in resultados.items():
        print(f"{nome:<10} {dados['tempo_carga']:<12.2f} {dados['tempo_transcricao']:<18.2f} {dados['texto'][:40]}")

    return resultados

# Documentação das diferenças observadas:
#
# MODELO TINY (~39M parâmetros):
#   - Mais rápido para carregar e transcrever
#   - Menor precisão, especialmente com sotaques ou ruído de fundo
#   - Pode errar palavras ou perder contexto em frases longas
#   - Ideal para aplicações que priorizam velocidade
#
# MODELO BASE (~74M parâmetros):
#   - Equilíbrio entre velocidade e precisão
#   - Boa precisão para falas claras em português
#   - Tempo de processamento moderado
#   - Recomendado para uso geral
#
# MODELO SMALL (~244M parâmetros):
#   - Mais lento para carregar e transcrever
#   - Maior precisão, melhor com ruído e sotaques variados
#   - Captura melhor pontuação e contexto
#   - Ideal para aplicações que priorizam precisão

print("Função testar_modelos_whisper definida com sucesso!")
print("Para testar, execute: testar_modelos_whisper('audio.wav')")

### Exercício 4: Tratamento de Exceções em Arquivos
Atualize a função `transcrever_audio` para verificar se o arquivo `audio.wav` realmente existe no diretório antes de chamar o modelo. Caso o arquivo não seja encontrado, a função deve retornar uma mensagem amigável de erro em vez de quebrar a execução.

In [ ]:
import whisper
import os

def transcrever_audio(arquivo="audio.wav"):
    """
    Transcreve o áudio de um arquivo WAV usando o modelo Whisper.
    Verifica a existência do arquivo antes de processar e trata exceções.
    """
    # Verificar se o arquivo existe
    if not os.path.exists(arquivo):
        mensagem = f"Arquivo '{arquivo}' não encontrado no diretório atual ({os.getcwd()})."
        print(f"⚠ Erro: {mensagem}")
        print("Dica: Grave um áudio primeiro usando a função gravar_audio().")
        return None

    # Verificar se o arquivo não está vazio
    if os.path.getsize(arquivo) == 0:
        print(f"⚠ Erro: O arquivo '{arquivo}' está vazio.")
        return None

    try:
        print("Transcrevendo...")
        modelo = whisper.load_model("base")
        resultado = modelo.transcribe(arquivo, language="pt")
        texto = resultado["text"]
        print(f"Texto reconhecido: {texto}")
        return texto

    except Exception as e:
        print(f"⚠ Erro durante a transcrição: {e}")
        print("Verifique se o arquivo de áudio é válido e tente novamente.")
        return None

# Testes da função
print("--- Teste 1: Arquivo inexistente ---")
resultado1 = transcrever_audio("arquivo_que_nao_existe.wav")
print(f"Retorno: {resultado1}")

print("\n--- Teste 2: Arquivo real (se existir) ---")
resultado2 = transcrever_audio("audio.wav")
print(f"Retorno: {resultado2}")

### Exercício 5: Evolução do Motor de Respostas (PLN)
A função atual `gerar_resposta` usa correspondência exata de substrings (`if "oi" in texto`). Substitua essa lógica implementando a biblioteca `spaCy`. Utilize o processamento do spaCy para identificar a intenção da frase através de lematização e análise de palavras-chave, tornando o assistente capaz de entender variações como "olá", "saudações" ou "qual o seu nome?".

In [ ]:
import spacy
import time

# Carregar o modelo de português do spaCy
nlp = spacy.load("pt_core_news_sm")

# Definição de intenções com lemas e palavras-chave associadas
INTENCOES = {
    "saudacao": {
        "lemas": ["oi", "olá", "saudação", "bom", "boa", "ei", "alô"],
        "palavras": ["oi", "olá", "saudações", "bom dia", "boa tarde", "boa noite", "e aí", "hey", "alô"],
        "resposta": "Olá! Como posso ajudar você?"
    },
    "nome": {
        "lemas": ["nome", "chamar", "querer", "identificar"],
        "palavras": ["nome", "chama", "quem é você", "se chama", "seu nome", "como te chamam"],
        "resposta": "Eu sou um assistente virtual feito em Python."
    },
    "hora": {
        "lemas": ["hora", "horário", "tempo", "relógio"],
        "palavras": ["hora", "horas", "horário", "que horas", "relógio"],
        "resposta": None  # Será gerada dinamicamente
    },
    "despedida": {
        "lemas": ["tchau", "adeus", "até", "despedida"],
        "palavras": ["tchau", "adeus", "até logo", "até mais", "falou", "bye"],
        "resposta": "Até logo! Foi um prazer ajudar."
    },
    "agradecimento": {
        "lemas": ["obrigar", "agradecer", "valeu"],
        "palavras": ["obrigado", "obrigada", "agradeço", "valeu", "thanks"],
        "resposta": "De nada! Estou aqui para ajudar."
    }
}

def gerar_resposta(texto):
    """
    Gera uma resposta utilizando spaCy para identificar a intenção
    através de lematização e análise de palavras-chave.
    """
    texto_lower = texto.lower().strip()
    doc = nlp(texto_lower)

    # Extrair lemas dos tokens (ignorando stopwords e pontuação)
    lemas = [token.lemma_ for token in doc if not token.is_punct]

    print(f"  [DEBUG] Texto: '{texto_lower}'")
    print(f"  [DEBUG] Lemas extraídos: {lemas}")

    # Verificar cada intenção
    melhor_intencao = None
    melhor_score = 0

    for intencao, dados in INTENCOES.items():
        score = 0

        # Verificar correspondência por lemas
        for lema in lemas:
            if lema in dados["lemas"]:
                score += 2  # Lemas têm peso maior

        # Verificar correspondência por palavras-chave no texto original
        for palavra in dados["palavras"]:
            if palavra in texto_lower:
                score += 1

        if score > melhor_score:
            melhor_score = score
            melhor_intencao = intencao

    # Gerar resposta baseada na intenção identificada
    if melhor_intencao and melhor_score > 0:
        print(f"  [DEBUG] Intenção detectada: {melhor_intencao} (score={melhor_score})")

        if melhor_intencao == "hora":
            return time.strftime("Agora são %H horas e %M minutos.")
        else:
            return INTENCOES[melhor_intencao]["resposta"]
    else:
        return "Desculpe, ainda não sei responder isso."

# Testes demonstrativos
testes = [
    "Oi, tudo bem?",
    "Olá!",
    "Saudações, amigo",
    "Qual o seu nome?",
    "Como você se chama?",
    "Que horas são?",
    "Me diga o horário",
    "Tchau, até mais!",
    "Muito obrigado!",
    "Qual a capital do Brasil?"
]

print("=" * 60)
print("TESTES DO MOTOR DE RESPOSTAS COM spaCy")
print("=" * 60)

for frase in testes:
    print(f"\nEntrada: '{frase}'")
    resposta = gerar_resposta(frase)
    print(f"Resposta: {resposta}")
    print("-" * 40)

### Exercício 6: Assistente Contínuo
A função `assistente()` atualmente executa apenas um ciclo de gravação e resposta. Refatore essa função inserindo um laço `while True` para que o assistente funcione continuamente. Implemente uma condição de parada: se a transcrição contiver a palavra "desligar" ou "encerrar", o laço deve ser interrompido e o programa finalizado.

In [ ]:
import sounddevice as sd
import scipy.io.wavfile as wav
import whisper
import pyttsx3
import time
import os

def gravar_audio_continuo(nome_arquivo="audio.wav", duracao=5, fs=44100):
    """Grava áudio com tratamento de erro."""
    print("\nPreparando para gravar...")
    for i in range(3, 0, -1):
        print(i)
        time.sleep(1)
    try:
        print("Gravando...")
        audio = sd.rec(int(duracao * fs), samplerate=fs, channels=1)
        sd.wait()
        wav.write(nome_arquivo, fs, audio)
        print("Gravação finalizada!")
        return True
    except Exception as e:
        print(f"Erro na gravação: {e}")
        return False

def transcrever_audio_continuo(arquivo="audio.wav"):
    """Transcreve áudio com verificação de arquivo."""
    if not os.path.exists(arquivo):
        print(f"Arquivo '{arquivo}' não encontrado.")
        return ""
    print("Transcrevendo...")
    modelo = whisper.load_model("base")
    resultado = modelo.transcribe(arquivo, language="pt")
    texto = resultado["text"]
    print(f"Texto reconhecido: {texto}")
    return texto

def gerar_resposta_continuo(texto):
    """Gera resposta simples baseada em palavras-chave."""
    texto = texto.lower()
    if "oi" in texto or "olá" in texto:
        return "Olá! Como posso ajudar você?"
    elif "nome" in texto:
        return "Eu sou um assistente virtual feito em Python."
    elif "hora" in texto:
        return time.strftime("Agora são %H horas e %M minutos.")
    else:
        return "Desculpe, ainda não sei responder isso."

def sintetizar_voz_continuo(texto):
    """Sintetiza voz com pyttsx3."""
    print("Gerando fala...")
    engine = pyttsx3.init()
    engine.setProperty('rate', 180)
    engine.setProperty('volume', 1.0)
    voices = engine.getProperty('voices')
    for voice in voices:
        if "brazil" in voice.name.lower() or "portuguese" in voice.name.lower():
            engine.setProperty('voice', voice.id)
            break
    engine.say(texto)
    engine.runAndWait()

def assistente_continuo():
    """
    Assistente virtual que funciona continuamente em loop.
    Para encerrar, diga 'desligar' ou 'encerrar'.
    """
    print("=" * 50)
    print("ASSISTENTE VIRTUAL CONTÍNUO")
    print("Diga 'desligar' ou 'encerrar' para sair.")
    print("=" * 50)

    ciclo = 1

    while True:
        print(f"\n--- Ciclo {ciclo} ---")

        # 1. Gravar áudio
        if not gravar_audio_continuo():
            print("Falha na gravação. Tentando novamente...")
            continue

        # 2. Transcrever áudio
        texto = transcrever_audio_continuo("audio.wav")

        if not texto:
            print("Nenhum texto detectado. Tentando novamente...")
            ciclo += 1
            continue

        # 3. Verificar condição de parada
        texto_lower = texto.lower()
        if "desligar" in texto_lower or "encerrar" in texto_lower:
            mensagem_despedida = "Encerrando o assistente. Até logo!"
            print(f"\nResposta: {mensagem_despedida}")
            sintetizar_voz_continuo(mensagem_despedida)
            print("\n" + "=" * 50)
            print("Assistente encerrado com sucesso!")
            print("=" * 50)
            break

        # 4. Gerar e falar resposta
        resposta = gerar_resposta_continuo(texto)
        print(f"Resposta: {resposta}")
        sintetizar_voz_continuo(resposta)

        ciclo += 1

# Para executar o assistente contínuo, descomente a linha abaixo:
# assistente_continuo()

print("Função assistente_continuo definida com sucesso!")
print("Para iniciar, execute: assistente_continuo()")
print("Diga 'desligar' ou 'encerrar' para parar o assistente.")

### Exercício 7: Personalização Dinâmica da Voz
Modifique a função `sintetizar_voz` para receber os parâmetros `rate` (velocidade) e `volume` como argumentos da função. Crie uma lógica que permita ao usuário escolher se deseja uma voz masculina ou feminina, iterando sobre a propriedade `voices` e selecionando o ID correspondente.

In [ ]:
import pyttsx3

def sintetizar_voz(texto, rate=180, volume=1.0, genero="feminina"):
    """
    Sintetiza voz com personalização dinâmica de velocidade, volume e gênero.

    Parâmetros:
        texto (str): Texto a ser falado.
        rate (int): Velocidade da fala (palavras por minuto). Padrão: 180.
        volume (float): Volume da fala (0.0 a 1.0). Padrão: 1.0.
        genero (str): 'masculina' ou 'feminina'. Padrão: 'feminina'.
    """
    print(f"Gerando fala...")
    print(f"  Configuração: rate={rate} | volume={volume} | gênero={genero}")

    engine = pyttsx3.init()

    # Configurar velocidade e volume
    engine.setProperty('rate', rate)
    engine.setProperty('volume', volume)

    # Listar vozes disponíveis
    voices = engine.getProperty('voices')

    print(f"\n  Vozes disponíveis no sistema:")
    for i, voice in enumerate(voices):
        print(f"    [{i}] {voice.name} (ID: {voice.id})")

    # Selecionar voz baseada no gênero
    voz_selecionada = None
    genero_lower = genero.lower()

    for voice in voices:
        nome_lower = voice.name.lower()

        if genero_lower == "feminina" or genero_lower == "f":
            # Vozes femininas comuns no Windows: Maria (PT-BR), Zira (EN-US)
            if any(nome in nome_lower for nome in ["maria", "zira", "female", "sabina", "helena"]):
                voz_selecionada = voice
                break
        elif genero_lower == "masculina" or genero_lower == "m":
            # Vozes masculinas comuns no Windows: David (EN-US), Daniel
            if any(nome in nome_lower for nome in ["david", "daniel", "male", "mark"]):
                voz_selecionada = voice
                break

    if voz_selecionada:
        engine.setProperty('voice', voz_selecionada.id)
        print(f"\n  Voz selecionada: {voz_selecionada.name}")
    else:
        print(f"\n  Voz {genero} não encontrada. Usando a primeira voz disponível: {voices[0].name}")
        engine.setProperty('voice', voices[0].id)

    engine.say(texto)
    engine.runAndWait()
    print("  Fala concluída!")

# Demonstração com diferentes configurações
print("=" * 50)
print("TESTE 1: Voz feminina, velocidade normal")
print("=" * 50)
sintetizar_voz("Olá! Eu sou a voz feminina do assistente.", rate=180, volume=1.0, genero="feminina")

print("\n" + "=" * 50)
print("TESTE 2: Voz feminina, velocidade lenta")
print("=" * 50)
sintetizar_voz("Agora estou falando mais devagar.", rate=120, volume=0.8, genero="feminina")

print("\n" + "=" * 50)
print("TESTE 3: Voz feminina, velocidade rápida")
print("=" * 50)
sintetizar_voz("E agora estou falando bem rápido!", rate=250, volume=1.0, genero="feminina")

### Exercício 8: Análise de Dependências do SO
Na primeira célula de instalação, são incluídas as bibliotecas `pywin32` e `comtypes` juntamente com o `pyttsx3`. Pesquise e explique por que essas dependências específicas são necessárias para o funcionamento da síntese de voz em sistemas operacionais Windows.

In [ ]:
# ============================================================================
# EXERCÍCIO 8 - Análise de Dependências do SO
# ============================================================================
#
# Por que pywin32 e comtypes são necessárias para pyttsx3 no Windows?
#
# --------------------------------------------------------------------------
# 1. PYTTSX3 E O SAPI5 (Speech API 5)
# --------------------------------------------------------------------------
# O pyttsx3 é uma biblioteca de síntese de voz offline e multiplataforma.
# Em cada sistema operacional, ele utiliza um motor de voz diferente:
#   - Windows: SAPI5 (Speech Application Programming Interface 5)
#   - macOS:   NSSpeechSynthesizer
#   - Linux:   espeak
#
# O SAPI5 é a API nativa da Microsoft para síntese e reconhecimento de voz,
# e é acessado internamente através da tecnologia COM (Component Object Model).
#
# --------------------------------------------------------------------------
# 2. PYWIN32 (python for windows extensions)
# --------------------------------------------------------------------------
# O pywin32 fornece acesso às APIs nativas do Windows diretamente pelo
# Python. Ele inclui módulos como:
#   - win32com: permite interagir com objetos COM do Windows
#   - win32api: acesso às funções da API do Windows
#   - pywintypes: tipos de dados nativos do Windows
#
# O pyttsx3 depende do pywin32 porque precisa do módulo win32com para
# criar instâncias do motor SAPI5 (SpVoice). Sem o pywin32, o Python
# não conseguiria se comunicar com o sistema COM do Windows.
#
# Internamente, pyttsx3 faz chamadas como:
#   import win32com.client
#   self._tts = win32com.client.Dispatch('SAPI.SpVoice')
#
# Isso cria uma instância do motor de voz SAPI usando a infraestrutura COM.
#
# --------------------------------------------------------------------------
# 3. COMTYPES
# --------------------------------------------------------------------------
# O comtypes é uma biblioteca alternativa/complementar ao pywin32 para
# trabalhar com o COM (Component Object Model) do Windows.
#
# Ele permite:
#   - Acessar e criar objetos COM
#   - Chamar interfaces COM de baixo nível
#   - Gerar wrappers Python para bibliotecas COM automaticamente
#
# O pyttsx3 utiliza o comtypes como fallback ou complemento ao pywin32
# para garantir a comunicação COM, especialmente em cenários onde
# determinadas interfaces COM precisam de um acesso mais direto.
#
# --------------------------------------------------------------------------
# 4. RESUMO
# --------------------------------------------------------------------------
# A cadeia de dependências funciona assim:
#
#   pyttsx3  --->  pywin32 (win32com.client)  --->  SAPI5 (COM)  --->  Voz do Windows
#              |                                       ^
#              +--> comtypes (acesso COM direto) -------+
#
# Sem essas bibliotecas, o pyttsx3 não conseguiria acessar o motor de
# síntese de voz do Windows, resultando em erros como:
#   - ModuleNotFoundError: No module named 'pywintypes'
#   - ModuleNotFoundError: No module named 'win32com'
#   - pyttsx3.EngineError: "sapi5" is not supported
#
# Em sistemas Linux ou macOS, essas dependências NÃO são necessárias,
# pois o pyttsx3 utiliza motores diferentes (espeak ou NSSpeechSynthesizer).
# ============================================================================

print("Resposta teórica documentada nos comentários acima.")
print()
print("Resumo:")
print("- pywin32: Fornece o módulo win32com.client para criar objetos COM")
print("  do Windows. O pyttsx3 usa isso para instanciar o motor SAPI5.")
print("- comtypes: Biblioteca complementar para acesso COM de baixo nível.")
print("  Serve como alternativa/complemento ao pywin32 para interfaces COM.")
print("- Ambas são necessárias APENAS no Windows, pois o motor de voz")
print("  utilizado (SAPI5) é baseado na tecnologia COM da Microsoft.")